In [1]:
import sys
from pathlib import Path

# Add project root to Python path
ROOT_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT_DIR))

import importlib
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from concurrent.futures import ThreadPoolExecutor
from typing import Literal
import src.evaluation as ev
importlib.reload(ev)
from src.ingest import load_machine_data, build_index

GROUND_TRUTH_OUTPUT_PATH = ROOT_DIR / "data"
RAG_ANSWER_PATH = ROOT_DIR / "data"
RAG_EVALUATE_PATH = ROOT_DIR / "data"

%load_ext autoreload
%autoreload 2

# 1. LLM Judgement
## 1.1 Reading RAG answer file

In [2]:
# load rag answer file 1
df_rag_answers1 = pd.read_csv(RAG_ANSWER_PATH / 'rag_answer.csv')
dic_rag_answers1 =  df_rag_answers1.to_dict(orient='records')

# load rag answer file 2
df_rag_answers2 = pd.read_csv(RAG_ANSWER_PATH / 'rag_answer2.csv')
dic_rag_answers2 =  df_rag_answers2.to_dict(orient='records')


In [3]:
# define output type answer format
class AnswerEvaluation(BaseModel):
    reasoning:str = Field(
        description = "Reasoning about the quality of the answer."
    )
    score : Literal['good','bad'] = Field(
        description= "'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [4]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a machine operator/equipment engineer/equipment technician
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [7]:
dic_rag_answers1[10]

{'question': 'What reading range on tool wear usually points to a tool wear failure, and is it a hard cutoff?',
 'llm_answer': 'A tool wear reading in the **200–240 min range** usually points to **increased risk of Tool Wear Failure**.\n\n- **What’s happening:** the cutting tool edge is progressively degrading, which reduces cutting ability and can eventually lead to fracture or process failure.\n- **Why:** this is a **wear-out failure** driven by time/usage accumulation, and the context says the **primary indicator** is `Tool wear [min]` in the **200–240** range.\n- **Is it a hard cutoff?** **No.** It is explicitly described as a **probabilistic band, not a deterministic threshold**. A unit in that range may still not have failed.\n\n**Recommended action:**\n- **Flag any unit with tool wear > 200 min for priority inspection.**\n- **Schedule tool replacement before 240 min** if the unit is expected to keep running.\n- **Do not use torque or rotational speed as early warning signs** for

In [8]:
dic_rag_answers2[10]

{'question': 'What reading range on tool wear usually points to a tool wear failure, and is it a hard cutoff?',
 'llm_answer': 'The reading range that usually points to tool wear failure is 200–240 minutes. However, it is not a hard cutoff; this range is probabilistic, meaning that tools can operate within it without failing. If a unit has tool wear above 200 minutes, it should be flagged for priority inspection and tool replacement should be scheduled before reaching 240 minutes.',
 'original_answer': '- Primary indicator: `Tool wear [min]` in the 200–240 range\n- Torque and rotational speed are **not** reliable leading indicators for TWF — this is the key diagnostic distinction from OSF and PWF\n- No air/process temperature anomaly expected\n\n**False-positive note:** A unit with tool wear in the 200–240 band that has NOT failed is common — this is a probabilistic band, not a deterministic threshold. Do not treat "in-band" as "failed."',
 'document': 'e8a1fe10fc'}

# 2. Evaluating First RAG answer file

In [ ]:
# build judgement prompt first
record1 = dic_rag_answers1[10]


In [24]:
# evaluate rag answer1
openai_client = OpenAI()
model="gpt-5.4-mini"

def judge_rag_answer1(record1):

    eval_result, usage = ev.evaluate_rag_answer(
        aqa_judge_prompt,   
        record1,
        openai_client,
        aqa_judge_instructions,
        AnswerEvaluation,
        model
    )
    
    judgement_result = {
        'question': record1['question'],
        'document' : record1['document'],
        'score' : eval_result.score,
        'reasoning' : eval_result.reasoning
    }

    return judgement_result, usage

In [25]:
with ThreadPoolExecutor(max_workers=6) as pool:
    judgement_results1 = ev.map_progress(pool, dic_rag_answers1, judge_rag_answer1)

  0%|          | 0/355 [00:00<?, ?it/s]

In [26]:
evaluations = []
usages = []

for eval, usage in judgement_results1:
    evaluations.append(eval)
    usages.append(usage)

print(f"Total price = {ev.calculate_total_price(usages)}")
df_eval1 = pd.DataFrame(evaluations)

Total price = 0.31364925000000005


## 2.1 Save RAG Evaluating file 1

In [27]:
# save evaluation score file
df_eval1.to_csv(RAG_EVALUATE_PATH / "rag_evaluate1.csv")

In [28]:
df_eval1['score'].unique()

<ArrowStringArray>
['good', 'bad']
Length: 2, dtype: str

# 3. Evaluating Second RAG answer file

In [29]:
record2 = dic_rag_answers2[10]


In [30]:
# evaluate rag answer2
openai_client = OpenAI()
model="gpt-5.4-mini"

def judge_rag_answer2(record2):

    # get evaluate result
    eval_result, usage = ev.evaluate_rag_answer(aqa_judge_prompt,
                                            record2,
                                            openai_client,
                                            aqa_judge_instructions,
                                            AnswerEvaluation,
                                            model)
    
    judgement_result = {
        'question': record2['question'],
        'document' : record2['document'],
        'score' : eval_result.score,
        'reasoning' : eval_result.reasoning
    }

    return judgement_result, usage

In [31]:
with ThreadPoolExecutor(max_workers=6) as pool:
    judgement_results2 = ev.map_progress(pool, dic_rag_answers2, judge_rag_answer2)

  0%|          | 0/355 [00:00<?, ?it/s]

In [32]:
evaluations = []
usages = []

for eval, usage in judgement_results2:
    evaluations.append(eval)
    usages.append(usage)

print(f"Total price = {ev.calculate_total_price(usages)}")
df_eval2 = pd.DataFrame(evaluations)

Total price = 0.26601600000000003


## 3.1 Save RAG Evaluating file 2

In [33]:
# save evaluation score file
df_eval2.to_csv(RAG_EVALUATE_PATH / "rag_evaluate2.csv")

In [34]:
df_eval2['score'].unique()

<ArrowStringArray>
['good', 'bad']
Length: 2, dtype: str

# 4. Evaluting Summary

In [35]:
def summarize(df):
    counts = df['score'].value_counts(normalize=True) * 100
    return counts.get('good', 0)  # just the % good

pct_good_a = summarize(df_eval1)
pct_good_b = summarize(df_eval2)

print(f"Approach A: {pct_good_a:.1f}% good")
print(f"Approach B: {pct_good_b:.1f}% good")

Approach A: 82.8% good
Approach B: 79.7% good


So we can consider Approach A  giving better result which use gpt-5.4-mini and complete prompt template while Approach B was the shorter prompt + gpt-4o-mini.